In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks
import os
# 3. Time Warp (esticando/comprimindo tempo)
from scipy.interpolate import interp1d
import ecgmentations as E
import random

In [2]:
# Caminho do CSV de um paciente 
csv_file = "mitbih_all_records_renumerada.csv"

# Carrega os dados
df = pd.read_csv(csv_file)

In [3]:
# Visualiza as primeiras colunas disponíveis
print(df.head())

   channel_0  channel_1  sample # type       record
0     -0.145     -0.065         0   NC  100_labeled
1     -0.145     -0.065         1   NC  100_labeled
2     -0.145     -0.065         2   NC  100_labeled
3     -0.145     -0.065         3   NC  100_labeled
4     -0.145     -0.065         4   NC  100_labeled


In [4]:
def get_beat_segment_by_similar_amplitude_2(df, desired_type="V", nth=0, channel="channel_0", amplitude_tolerance=0.2):
    """
    Localiza um batimento e retorna os dados (amostras e amplitudes) do segmento
    delimitado por picos de amplitude similar.

    Retorna:
    - Um DataFrame do pandas com os dados do segmento.
    - None se o batimento ou um segmento válido não for encontrado.
    """
    
    df = df.sort_values("sample #").reset_index(drop=True)

    target_rows = df[df["type"] == desired_type].reset_index(drop=True)
    if not len(target_rows):
        print(f"Aviso: Nenhum batimento do tipo '{desired_type}' encontrado.")
        return None
    
    nth = min(nth, len(target_rows) - 1)
    center_sample = int(target_rows.loc[nth, "sample #"])

    x = df["sample #"].values
    y = df[channel].values

    peaks, _ = find_peaks(y, distance=100)
    if len(peaks) < 2:
        print("Aviso: Menos de dois picos R detectados.")
        return None

    center_peak_index = np.argmin(np.abs(peaks - center_sample))
    center_peak_sample = peaks[center_peak_index]
    center_peak_amplitude = y[center_peak_sample]

    min_amp = center_peak_amplitude * (1 - amplitude_tolerance)
    max_amp = center_peak_amplitude * (1 + amplitude_tolerance)

    # Localiza picos anteriores e posteriores com amplitude semelhante
    start_peak = None
    for i in range(center_peak_index - 1, -1, -1):
        if min_amp <= y[peaks[i]] <= max_amp:
            start_peak = peaks[i]
            break
    if start_peak is None:
        start_peak = peaks[0]

    end_peak = None
    for i in range(center_peak_index + 1, len(peaks)):
        if min_amp <= y[peaks[i]] <= max_amp:
            end_peak = peaks[i]
            break
    if end_peak is None:
        end_peak = peaks[-1]

    # Cria a máscara e extrai o segmento completo (mantendo todas as colunas originais)
    mask = (df["sample #"] >= start_peak) & (df["sample #"] <= end_peak)
    segment_df = df.loc[mask].copy()  # <— agora mantém TODAS as colunas

    # Opcional: renomeia a coluna do canal atual para "amplitude"
    # (mantendo as demais colunas intactas)
    if channel in segment_df.columns:
        segment_df.rename(columns={channel: "amplitude"}, inplace=True)
    
    
    return segment_df


In [5]:
def plot_beat_segment(segment_df, title="Segmento de Batimento Cardíaco"):
    """
    Função auxiliar para plotar um segmento de batimento de um DataFrame.
    """
    if segment_df is None or segment_df.empty:
        print("DataFrame vazio. Nada para plotar.")
        return

    plt.figure(figsize=(12, 6))
    plt.plot(segment_df["sample #"], segment_df["amplitude"], label="Sinal")
    plt.title(title)
    plt.xlabel("Número da Amostra")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

In [6]:
def save_augmented(segment_df, save_path, aug_name):
    """Salva segmento em CSV no formato padrão."""
    augmented_df = segment_df.copy()
    augmented_df["channel_0"] = augmented_df["amplitude"]
    augmented_df = augmented_df[["channel_0", "sample #", "type"]]
    # adiciona coluna type como "AUG"
    #augmented_df["type"] = aug_name
    augmented_df.to_csv(save_path, index=False)

In [7]:
def augment_add_sine_pulse(segment_df, amplitude=0.1, frequency=1.0, random_phase=True, save_path="aug_sinepulse.csv"):
    """
    Adiciona uma onda senoidal ao sinal de ECG para simular deriva da linha de base.

    Parâmetros:
    - segment_df (pd.DataFrame): DataFrame com as colunas 'sample #' e 'amplitude'.
    - amplitude (float): A amplitude (altura) máxima da onda senoidal a ser adicionada.
    - frequency (float): O número de ciclos completos da onda senoidal ao longo do segmento.
    - random_phase (bool): Se True, a onda senoidal começará em um ponto aleatório do seu ciclo.
    - save_path (str): Caminho opcional para salvar o DataFrame aumentado.

    Retorna:
    - pd.DataFrame: O DataFrame com a onda senoidal adicionada.
    """
    augmented_df = segment_df.copy()
    
    y = segment_df["amplitude"].values
    
    # Cria um eixo de tempo de 0 ao comprimento do sinal
    x = np.arange(len(y))
    
    # Define a fase inicial da onda (pode ser aleatória ou zero)
    phase = np.random.uniform(0, 2 * np.pi) if random_phase else 0
    
    # Gera a onda senoidal
    # A fórmula garante que o número de 'frequency' ciclos completos se encaixe no comprimento do sinal
    sine_wave = amplitude * np.sin(2 * np.pi * frequency * (x / len(x)) + phase)
    
    # Adiciona a onda senoidal ao sinal original
    augmented_df["amplitude"] = y + sine_wave
    
    # Salva o resultado (opcional)
    save_augmented(augmented_df, save_path, f"sine_amp{amplitude}_freq{frequency}")
    
    return augmented_df

In [ ]:

for j in range(4614,8000):
        augment_add_sine_pulse(get_beat_segment_by_similar_amplitude_2(df, desired_type="L", nth=j, channel="channel_0", amplitude_tolerance = 0.7), amplitude = random.uniform(0.1, 0.05), frequency = random.uniform(1.0, 0.5), save_path = f"data_aug/aug_sinepulse{j}.csv")
        print((j/8000)*100)